# Analysis of nonlocality of sparse-autoencoder features

## Statistics of feature sparsity

In this section we will analyze the nonlocality of each feature by computing how many tokens activate each feature, and study the statistics. 

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# Load the saved data

site='resid_out_layer0'
data = torch.load(f"feature_sparsity_data_{site}.pt")
frequencies = data["frequencies"]
THRESHOLD = data["threshold"]

# Filter out dead features
active_freqs = frequencies[frequencies > 0].cpu().numpy()

counts, bin_edges = np.histogram(active_freqs, bins=200)

plt.figure(figsize=(10, 6))
#plt.hist(active_freqs, bins=200, log=True, color='skyblue', edgecolor='black')
#plt.title(f"Feature Activation Frequency Distribution (Threshold > {THRESHOLD})")

plt.plot(bin_edges[1:], 1/counts,'r.-')
plt.xlabel("Activation Probability")
plt.ylabel("Count of Features (Log Scale)")
#plt.grid(True, which="both", ls="-", alpha=0.2)
plt.show()

print(counts)

In [ ]:
plt.figure()
plt.plot(np.log(np.sort(active_freqs)))
plt.show()

In [ ]:
# Compute number of unique tokens for each feature.

import torch
import matplotlib.pyplot as plt

# 1. Load the data
site='resid_out_layer0'
data = torch.load(f"feature_sparsity_data_{site}.pt")
feature_token_counts = data["feature_token_counts"] # This is a list of Counter objects

# 2. Count unique tokens for each feature
# len(counter) gives the number of unique keys (distinct tokens)
unique_counts = [len(counter) for counter in feature_token_counts]

# 3. Filter out dead features (those with 0 activations)
active_unique_counts = [c for c in unique_counts if c > 0]

counts, bin_edges = np.histogram(active_unique_counts, bins=50)

# 4. Plot the distribution
plt.figure(figsize=(10, 6))
plt.hist(active_unique_counts, bins=100, log=True, color='lightgreen', edgecolor='black')
plt.title("Distribution of Unique Tokens per Feature")
plt.xlabel("Number of Unique Tokens Triggering the Feature")
plt.ylabel("Count of Features (Log Scale)")
plt.grid(True, which="both", ls="-", alpha=0.2)
plt.show()

# Optional: Print some stats
print(f"Max unique tokens for a single feature: {max(unique_counts)}")
print(f"Average unique tokens (for active features): {sum(active_unique_counts)/len(active_unique_counts):.2f}")

In [ ]:
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

# 2. Filter non-zero bins for log fitting
valid_mask = (counts > 0) & (bin_centers < 10000)
x_data = bin_centers[valid_mask]
y_data = counts[valid_mask]

# 3. Fit Power Law: log(y) = alpha * log(x) + C
# This fits the relationship y = exp(C) * x^alpha
inv_x=1/x_data
log_y = np.log(y_data)

# Fit linear model in log-log space
slope, intercept = np.polyfit(inv_x, log_y, 1)
fitted_y = intercept+inv_x*slope

plt.figure()
plt.plot(bin_edges[1:], np.log(counts),'ro')
plt.plot(x_data, fitted_y,'b-')
plt.show()

The result suggests the following formula:
$$\rho(P) = Ce^{-\mu/P}$$

$\rho(P)$ is the number of features active at a given activation probability $P$.


## Visualize correlation between features

In this section we compute the correlation between features and visualize the results. The correlation is defined by 

$C_{nm}\equiv \left\langle W_nW_m\right\rangle-\left\langle W_n\right\rangle\left\langle W_m\right\rangle$

where the average is taken in
the ensemble of tokens in the provided training text. The correlation matrix is computed by `compute_correlations.py` and saved as `correlation_matrix.pt`.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import MDS
from sklearn.manifold import TSNE

# Load correlation matrix
data = torch.load("correlation_matrix.pt")
cov_matrix = data["covariance_matrix"]
leading_features = data["leading_features"]

# Convert covariance to correlation
diag = torch.sqrt(torch.diag(cov_matrix))
denom = torch.outer(diag, diag)
corr_matrix = cov_matrix / (denom + 1e-10)

# Compute distance matrix: D = -log(|Corr|)
abs_corr = torch.abs(corr_matrix)
epsilon = 1e-6
abs_corr = torch.clamp(abs_corr, min=epsilon, max=1.0)
dist_matrix = -torch.log(abs_corr)

dist_from_corr = 1 + 1e-5 - corr_matrix.numpy()
tsne = TSNE(n_components=2, metric='precomputed', init='random', random_state=42)
embedding = tsne.fit_transform(dist_from_corr)
# Plot
plt.figure(figsize=(14, 12))

# Draw correlation links
correlation_threshold = 0.2  # Adjust this threshold as needed
max_linewidth = 3.0

# Get correlation matrix as numpy array
corr_np = corr_matrix.numpy()

# Draw lines for correlations above threshold
for i in range(len(leading_features)):
    for j in range(i+1, len(leading_features)):
        corr_value = abs(corr_np[i, j])
        if corr_value > correlation_threshold:
            # Line thickness proportional to correlation strength
            linewidth = (corr_value - correlation_threshold) / (1 - correlation_threshold) * max_linewidth
            # Color: red for positive, blue for negative
            color = 'red' if corr_np[i, j] > 0 else 'blue'
            alpha = 0.3 + 0.4 * (corr_value - correlation_threshold) / (1 - correlation_threshold)
            plt.plot([embedding[i, 0], embedding[j, 0]], 
                    [embedding[i, 1], embedding[j, 1]], 
                    color=color, linewidth=linewidth, alpha=alpha, zorder=1)

# Plot points on top of lines
plt.scatter(embedding[:, 0], embedding[:, 1], alpha=0.8, c='darkgreen', 
           edgecolors='k', s=100, zorder=2)

# Annotate points
for i, (x, y) in enumerate(embedding):
    plt.annotate(str(leading_features[i]), (x, y), fontsize=9, alpha=0.9, 
                ha='center', va='center', zorder=3)

plt.title(f"Feature Correlation Map (threshold={correlation_threshold})")
plt.xlabel("MDS Dimension 1")
plt.ylabel("MDS Dimension 2")
plt.grid(True, alpha=0.2)

# Add legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color='red', lw=2, label='Positive correlation'),
                  Line2D([0], [0], color='blue', lw=2, label='Negative correlation')]
plt.legend(handles=legend_elements, loc='best')

plt.show()

### Plotting the feature importance and feature-feature correlation

In [ ]:
import torch

from sklearn.manifold import MDS
from sklearn.manifold import TSNE

import plotly.graph_objects as go
import numpy as np

# Load feature token counts to get unique token counts
data = torch.load("feature_sparsity_data.pt")
feature_token_counts = data["feature_token_counts"]

# Get unique token counts for leading features
unique_token_counts = []
for feat_idx in leading_features:
    unique_token_counts.append(len(feature_token_counts[feat_idx]))
unique_token_counts = np.array(unique_token_counts)

# Create figure
fig = go.Figure()

# Draw correlation links
correlation_threshold = 0.2
corr_np = corr_matrix.numpy()

# Collect line segments
for i in range(len(leading_features)):
    for j in range(i+1, len(leading_features)):
        corr_value = abs(corr_np[i, j])
        if corr_value > correlation_threshold:
            color = 'red' if corr_np[i, j] > 0 else 'blue'
            width = (corr_value - correlation_threshold) / (1 - correlation_threshold) * 5
            
            fig.add_trace(go.Scatter3d(
                x=[embedding[i, 0], embedding[j, 0]],
                y=[embedding[i, 1], embedding[j, 1]],
                z=[unique_token_counts[i], unique_token_counts[j]],
                mode='lines',
                line=dict(color=color, width=width),
                opacity=0.4,
                showlegend=False,
                hoverinfo='skip'
            ))

# Add scatter points
fig.add_trace(go.Scatter3d(
    x=embedding[:, 0],
    y=embedding[:, 1],
    z=unique_token_counts,
    mode='markers+text',
    marker=dict(
        size=3,
        color=unique_token_counts,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Unique Tokens"),
        line=dict(color='black', width=1)
    ),
    text=[str(f) for f in leading_features],
    textposition='top center',
    textfont=dict(size=9),
    name='Features',
    hovertemplate='Feature: %{text}<br>Unique Tokens: %{z}<extra></extra>'
))

# Update layout
fig.update_layout(
    title=f'Interactive 3D Feature Correlation Map (threshold={correlation_threshold})',
    scene=dict(
        xaxis_title='Embedding Dimension 1',
        yaxis_title='Embedding Dimension 2',
        zaxis_title='Unique Tokens'
    ),
    width=1000,
    height=800
)

fig.show()

In [ ]:
# Heatmap of correlation matrix
plt.figure(figsize=(12, 10))
plt.imshow(corr_matrix.numpy(), cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='Correlation Coefficient')
plt.title("Feature Correlation Matrix Heatmap")
plt.xlabel("Feature Index (in leading_features list)")
plt.ylabel("Feature Index (in leading_features list)")
plt.show()

np.max(corr_matrix.numpy())

In [ ]:
# plot histogram of the matrix element of corr_matrix
import matplotlib.pyplot as plt
import numpy as np

# Get all correlation values (excluding diagonal)
# Remove diagonal elements (which are 0 after -log(1) = 0)
mask = ~np.eye(corr_matrix.shape[0], dtype=bool)
corr_values = (corr_matrix.numpy())[mask]


# Create histogram
plt.figure(figsize=(10, 6))
plt.hist(corr_values, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Correlation Coefficient')
plt.ylabel('Frequency')
plt.title('Distribution of $C_{nm}$')
plt.grid(True, alpha=0.3)
plt.axvline(0, color='red', linestyle='--', linewidth=1, label='Zero correlation')
plt.legend()
plt.show()

# Print statistics
print(f"Mean correlation: {np.mean(corr_values):.4f}")
print(f"Median correlation: {np.median(corr_values):.4f}")
print(f"Std deviation: {np.std(corr_values):.4f}")
print(f"Min correlation: {np.min(corr_values):.4f}")
print(f"Max correlation: {np.max(corr_values):.4f}")

In [ ]:
# This is a one-time use block to generate the feature_location.csv file from the feature_location_data.pt file

import torch
import csv
from transformers import AutoTokenizer

# Load the data
data = torch.load("feature_location_data.pt")

# Load tokenizer for decoding tokens
MODEL_NAME = "EleutherAI/pythia-70m-deduped"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Extract data
feature_counts = data["feature_counts"]
frequencies = data["frequencies"]
feature_token_counts = data["feature_token_counts"]
feature_activations = data["feature_activations"]

print(f"Loaded data for {len(feature_activations)} features")
print(f"Total tokens processed: {data['total_tokens']}")

# Generate CSV
with open("feature_location.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["feature_idx", "count", "frequency", "num_records", "top_tokens", "avg_position", "position_std"])
    
    for idx, (cnt, freq) in enumerate(zip(feature_counts.tolist(), frequencies.tolist())):
        # Get top 5 tokens
        top_t = feature_token_counts[idx].most_common(5)
        # Format: "token1(count), token2(count)"
        top_t_str = ", ".join([f"{repr(tokenizer.decode([t]))}({c})" for t, c in top_t])
        
        # Calculate position statistics
        num_records = len(feature_activations[idx])
        if num_records > 0:
            positions = [rec['abs_pos'] for rec in feature_activations[idx]]
            avg_pos = sum(positions) / len(positions)
            # Calculate standard deviation
            if len(positions) > 1:
                variance = sum((p - avg_pos) ** 2 for p in positions) / len(positions)
                std_pos = variance ** 0.5
            else:
                std_pos = 0.0
        else:
            avg_pos = 0.0
            std_pos = 0.0
        
        writer.writerow([idx, int(cnt), f"{freq:.6f}", num_records, top_t_str, f"{avg_pos:.2f}", f"{std_pos:.2f}"])

print("CSV file 'feature_location.csv' created successfully!")

### Analyze the location of features versus their activation

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# Load the data
data = torch.load("feature_location_data.pt")

# Extract data
frequencies = data["frequencies"].numpy()
feature_activations = data["feature_activations"]
total_tokens = data["total_tokens"]

# Calculate average batch position for each feature
avg_batch_positions = []
active_frequencies = []
feature_indices = []

# Filter for features with significant activation (e.g., > 0.1% activation rate)
THRESHOLD = 0.001  # 0.1% - adjust this as needed

for idx in range(len(feature_activations)):
    if frequencies[idx] > THRESHOLD:
        num_records = len(feature_activations[idx])
        if num_records > 0:
            # Use batch_pos instead of abs_pos
            batch_positions = [rec['batch_pos'] for rec in feature_activations[idx]]
            avg_batch_pos = sum(batch_positions) / len(batch_positions)
            avg_batch_positions.append(avg_batch_pos)
            active_frequencies.append(frequencies[idx])
            feature_indices.append(idx)

print(f"Plotting {len(avg_batch_positions)} features with activation frequency > {THRESHOLD:.2%}")

# Create the plot
plt.figure(figsize=(12, 8))
scatter = plt.scatter(avg_batch_positions, active_frequencies, 
                     c=active_frequencies, cmap='viridis', 
                     alpha=0.6, s=50, edgecolors='black', linewidth=0.5)

plt.xlabel('Average Batch Position (0-127)', fontsize=12)
plt.ylabel('Activation Frequency', fontsize=12)
plt.title('Feature Activation Frequency vs Average Batch Position', fontsize=14, fontweight='bold')
plt.yscale('log')  # Log scale for better visualization
plt.grid(True, alpha=0.3, linestyle='--')
plt.colorbar(scatter, label='Activation Frequency')

# Optionally annotate top features
top_n = 10
top_indices = np.argsort(active_frequencies)[-top_n:]
for i in top_indices:
    plt.annotate(f'{feature_indices[i]}', 
                xy=(avg_batch_positions[i], active_frequencies[i]),
                xytext=(5, 5), textcoords='offset points',
                fontsize=8, alpha=0.7)

plt.tight_layout()
plt.savefig('batch_position_vs_frequency.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Plot saved as 'batch_position_vs_frequency.png'")
print(f"Batch position range: {min(avg_batch_positions):.1f} to {max(avg_batch_positions):.1f}")

### Position distribution for top features

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# Load the data
data = torch.load("feature_location_data.pt")

# Extract data
frequencies = data["frequencies"].numpy()
feature_activations = data["feature_activations"]
total_tokens = data["total_tokens"]

# Get top N most frequent features
TOP_N = 50
top_feature_indices = np.argsort(frequencies)[-TOP_N:][::-1]  # Descending order

# Batch size (should be 128 based on your script)
BATCH_SIZE = 128

# Create a matrix: rows = features, columns = batch positions
position_counts = np.zeros((TOP_N, BATCH_SIZE))

for i, feat_idx in enumerate(top_feature_indices):
    # Count activations at each batch position
    for rec in feature_activations[feat_idx]:
        batch_pos = rec['batch_pos']
        if 0 <= batch_pos < BATCH_SIZE:
            position_counts[i, batch_pos] += 1
    
    # Normalize by total activations for this feature to get frequency
    total_activations = position_counts[i].sum()
    if total_activations > 0:
        position_counts[i] /= total_activations

# Create the line plot
fig, ax = plt.subplots(figsize=(14, 8))

# Use a colormap for the lines
colors = plt.cm.tab20(np.linspace(0, 1, TOP_N))

# Plot each feature as a line
for i, feat_idx in enumerate(top_feature_indices):
    ax.plot(range(BATCH_SIZE), position_counts[i], 
            label=f'Feature {feat_idx} ({frequencies[feat_idx]:.2%})',
            linewidth=2, alpha=0.7, color=colors[i])

ax.set_xlabel('Batch Position', fontsize=12)
ax.set_ylabel('Activation Frequency', fontsize=12)
ax.set_ylim(0, 0.5)
ax.set_title(f'Top {TOP_N} Features: Activation Distribution Across Batch Positions', 
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('feature_position_lines.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Line plot saved as 'feature_position_lines.png'")

In [ ]:
# Create the line plot
fig, ax = plt.subplots(figsize=(14,10))

# Use a colormap for the lines
colors = plt.cm.tab20(np.linspace(0, 1, TOP_N))

# Plot each feature as a line
for i, feat_idx in enumerate(top_feature_indices):
    ax.plot(range(BATCH_SIZE), position_counts[i], 
            label=f'Feature {feat_idx} ({frequencies[feat_idx]:.2%})',
            linewidth=2, alpha=0.7, color=colors[i])

ax.set_xlabel('Batch Position', fontsize=12)
ax.set_ylabel('Activation Frequency', fontsize=12)
ax.set_ylim(0, 0.15)
ax.set_title(f'Top {TOP_N} Features: Activation Distribution Across Batch Positions', 
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('feature_position_lines.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Line plot saved as 'feature_position_lines.png'")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the data
data = torch.load('feature_token_influence.pt')
feature_influences = data['feature_influences']

# 2. Sort features by total influence magnitude
sorted_features = sorted(
    feature_influences.keys(), 
    key=lambda f: np.sum(feature_influences[f]['mean_influence']), 
    reverse=True
)

# 3. Select Top K features to plot
TOP_K = 10
features_to_plot = sorted_features[:TOP_K]

for feat_idx in features_to_plot:
    info = feature_influences[feat_idx]
    
    # Convert lists to numpy arrays
    mean_inf = np.array(info['mean_influence'])
    std_inf = np.array(info['std_influence'])
    n = info['num_samples']
    
    x = np.arange(len(mean_inf))
    
    # Create a new figure for each feature
    plt.figure(figsize=(10, 4))
    
    # Plot mean line with standard deviation shading
    line, = plt.plot(x, mean_inf, label='Mean Influence', marker='o', markersize=3, color='b')
    plt.fill_between(x, mean_inf - std_inf, mean_inf + std_inf, alpha=0.2, color='b', label='±1 Std Dev')

    plt.xlabel('Input Token Position (0=furthest, end=feature activation)')
    plt.ylabel('Influence Strength J')
    plt.title(f'Feature {feat_idx}: Influence Distribution (n={n} samples)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the data
data = torch.load('feature_token_influence.pt')
feature_influences = data['feature_influences']

# 2. Sort features by total influence magnitude
sorted_features = sorted(
    feature_influences.keys(), 
    key=lambda f: np.sum(feature_influences[f]['mean_influence']), 
    reverse=True
)

# 3. Select Top K features to plot
TOP_K = 20
features_to_plot = sorted_features[:TOP_K]

for feat_idx in features_to_plot:
    info = feature_influences[feat_idx]
    
    if 'all_influences' not in info:
        print(f"[WARN] Feature {feat_idx}: 'all_influences' missing. Re-run script.")
        continue

    # [num_samples, seq_len]
    all_traces = np.array(info['all_influences'])
    n_samples, seq_len = all_traces.shape
    
    # Create Heatmap
    plt.figure(figsize=(10, 6))
    
    # Use imshow for the 2D array
    # Aspect='auto' ensures it fills the figure properly regardless of matrix shape
    # Origin='lower' puts the first sample at the bottom
    im = plt.imshow(all_traces, aspect='auto', cmap='viridis', origin='lower')
    
    plt.colorbar(im, label='Influence Strength J')
    
    plt.xlabel('Input Token Position (0=furthest, end=feature activation)')
    plt.ylabel('Sample Index')
    plt.title(f'Feature {feat_idx}: Influence Heatmap (n={n_samples})')
    
    plt.tight_layout()
    plt.show()

### Measure the nonlocality of the influence

For each batch where a feature is activated, we can normalize the influence vector and obtain a probability distribution. Then we can compute the entropy of this distribution to measure the nonlocality of the influence. 

**1. Basic facts about data in `feature_token_influence.pt`**

In [ ]:
import torch

# Load the data
data = torch.load('feature_token_influence.pt')
feature_influences = data['feature_influences']

# Count total features
total_features = len(feature_influences)
print(f"Total features in feature_token_influence.pt: {total_features}")

# Count features with 'all_influences' data (needed for entropy calculation)
features_with_all_data = sum(1 for info in feature_influences.values() 
                             if 'all_influences' in info)
print(f"Features with 'all_influences' data: {features_with_all_data}")

# Show some additional info
if feature_influences:
    sample_sizes = [info['num_samples'] for info in feature_influences.values()]
    print(f"\nSample size statistics:")
    print(f"  Min samples: {min(sample_sizes)}")
    print(f"  Max samples: {max(sample_sizes)}")
    print(f"  Mean samples: {sum(sample_sizes)/len(sample_sizes):.1f}")

**2. Entropy distribution among batches**

In [ ]:
import scipy.stats

# Define number of features to analyze
TOP_K = 20
features_to_plot = sorted_features[:TOP_K]

for feat_idx in features_to_plot:
    plt.figure(figsize=(12, 7))
    info = feature_influences[feat_idx]
    
    if 'all_influences' not in info:
        continue

    # [num_samples, seq_len]
    all_traces = np.array(info['all_influences'])
    
    # 1. Normalize absolute influence to a probability distribution
    eps = 1e-12 
    abs_traces = np.abs(all_traces) + eps
    probs = abs_traces / np.sum(abs_traces, axis=1, keepdims=True)
    
    # 2. Compute Shannon Entropy (axis=1 is across tokens)
    entropies = scipy.stats.entropy(probs, axis=1,base=2)
    
    # 3. Sort entropies ascending for the line plot
    sorted_entropies = np.sort(entropies)
    
    # 4. Plot
    # We use a normalized x-axis (0 to 1) so features with different 
    # sample sizes can be compared directly
    x_axis = np.linspace(0, 1, len(sorted_entropies))
    plt.plot(x_axis, sorted_entropies, label=f'Feature {feat_idx}', linewidth=2)

    # Add reference line for Maximum possible entropy: log(seq_len)
    max_entropy = np.log2(all_traces.shape[1])
    plt.axhline(max_entropy, color='black', linestyle='--', alpha=0.5, label='Max Theoretical Entropy')

    plt.title('Sorted Shannon Entropy per Feature (Ascending)', fontsize=14)
    plt.xlabel('Fraction of Samples (Percentile)', fontsize=12)
    plt.ylabel('Shannon Entropy (bits)', fontsize=12)
    plt.legend(loc='lower right')
    plt.grid(True, which="both", ls="-", alpha=0.2)

    plt.tight_layout()
    plt.show()

**3. Entropy average value vs influence**

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats

# Define list of sites (layers) to analyze
sites = [
    "resid_out_layer0",
    "resid_out_layer1",
    "resid_out_layer2",
    "resid_out_layer3",
    "resid_out_layer4",
    "resid_out_layer5",
]

# Store average entropies for each layer
layer_entropies = {}
layer_labels = []

# Process each layer
for site in sites:
    try:
        # Load the data
        data = torch.load(f'feature_token_influence_{site}.pt')
        feature_influences = data['feature_influences']
        
        # Compute average entropy for each feature (leading features only)
        avg_entropies = []
        
        for feat_idx, info in feature_influences.items():
            if 'all_influences' not in info:
                continue
            
            # Get all influence traces
            all_traces = np.array(info['all_influences'])  # [num_samples, seq_len]
            
            # Normalize absolute influence to probability distribution
            eps = 1e-12
            abs_traces = np.abs(all_traces) + eps
            probs = abs_traces / np.sum(abs_traces, axis=1, keepdims=True)
            
            # Compute Shannon Entropy for each sample
            entropies = scipy.stats.entropy(probs, axis=1, base=2)
            
            # Average entropy across all samples
            avg_entropy = np.mean(entropies)
            avg_entropies.append(avg_entropy)
        
        if len(avg_entropies) > 0:
            layer_entropies[site] = np.array(avg_entropies)
            layer_labels.append(site)
            print(f"{site}: {len(avg_entropies)} features, entropy range: {np.min(avg_entropies):.3f} to {np.max(avg_entropies):.3f} bits")
        else:
            print(f"{site}: No features with 'all_influences' data")
            
    except FileNotFoundError:
        print(f"{site}: File not found, skipping")
    except Exception as e:
        print(f"{site}: Error - {e}")

# Create subplots arranged vertically
n_layers = len(layer_entropies)
fig, axes = plt.subplots(n_layers, 1, figsize=(10, 3*n_layers), sharex=True)

# Use a colormap to assign different colors to each layer
colors = plt.cm.tab10(np.linspace(0, 1, len(layer_labels)))

# Plot histograms for each layer on separate subplots
for i, (site, entropies) in enumerate(layer_entropies.items()):
    ax = axes[i] if n_layers > 1 else axes
    ax.hist(entropies, bins=30, alpha=0.7, color=colors[i], edgecolor='black', linewidth=0.5)
    ax.set_ylabel('Count', fontsize=10)
    ax.set_title(f'{site} (n={len(entropies)})', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')

# Set common x-axis label
axes[-1].set_xlabel('Average Entropy (bits)', fontsize=12)

# Add overall title
fig.suptitle('Histogram of Average Entropy for Leading Features Across All Layers', 
             fontsize=14, fontweight='bold', y=1.0)

plt.tight_layout()
plt.show()

# Print summary statistics
print(f"\nSummary Statistics:")
print(f"{'Layer':<20} | {'Features':<10} | {'Mean Entropy':<15} | {'Std Entropy':<15}")
print("-" * 70)
for site, entropies in layer_entropies.items():
    print(f"{site:<20} | {len(entropies):<10} | {np.mean(entropies):<15.3f} | {np.std(entropies):<15.3f}")


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats

site='resid_out_layer4'
# 1. Load the data
data = torch.load(f'feature_token_influence_{site}.pt')
feature_influences = data['feature_influences']

# 2. Compute average entropy and total influence for each feature
avg_entropies = []
total_influences = []
feature_indices = []

for feat_idx, info in feature_influences.items():
    if 'all_influences' not in info:
        continue
    
    # Get all influence traces
    all_traces = np.array(info['all_influences'])  # [num_samples, seq_len]
    
    # Normalize absolute influence to probability distribution
    eps = 1e-12
    abs_traces = np.abs(all_traces) + eps
    probs = abs_traces / np.sum(abs_traces, axis=1, keepdims=True)
    
    # Compute Shannon Entropy for each sample
    entropies = scipy.stats.entropy(probs, axis=1, base=2)
    
    # Average entropy across all samples
    avg_entropy = np.mean(entropies)
    
    # Total influence = sum of mean_influence
    total_influence = np.sum(info['mean_influence'])
    
    avg_entropies.append(avg_entropy)
    total_influences.append(total_influence)
    feature_indices.append(feat_idx)

# Convert to numpy arrays
avg_entropies = np.array(avg_entropies)
total_influences = np.array(total_influences)
feature_indices = np.array(feature_indices)

# 3. Sort by total influence to get top features
sort_idx = np.argsort(total_influences)[::-1]  # Descending order
sorted_entropies = avg_entropies[sort_idx]
sorted_influences = total_influences[sort_idx]
sorted_feature_indices = feature_indices[sort_idx]

# 4. Select top N features to plot (set to None to plot all features)
TOP_N = None  # Set to None to plot all features, or specify a number like 200
if TOP_N is None:
    top_entropies = sorted_entropies
    top_influences = sorted_influences
    top_feature_indices = sorted_feature_indices
else:
    top_entropies = sorted_entropies[:TOP_N]
    top_influences = sorted_influences[:TOP_N]
    top_feature_indices = sorted_feature_indices[:TOP_N]

# 5. Create the plot
plt.figure(figsize=(12, 8))
scatter = plt.scatter(top_influences, top_entropies, 
                     c=top_influences, cmap='viridis', 
                     alpha=0.6, s=50, edgecolors='black', linewidth=0.3)

plt.xlabel('Total Influence', fontsize=12)
plt.xscale('log')
plt.ylabel('Average Entropy (bits)', fontsize=12)
num_features_str = f'Top {TOP_N}' if TOP_N else 'All'
plt.title(f'Average Entropy vs Total Influence for {num_features_str} Features', 
          fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, linestyle='--')
plt.colorbar(scatter, label='Total Influence')

# Optionally annotate top features (only if not too many)
if len(top_feature_indices) <= 100:  # Only annotate if reasonable number
    annotate_count = min(15, len(top_feature_indices))
    for i in range(annotate_count):
        plt.annotate(f'{top_feature_indices[i]}', 
                    xy=(top_influences[i], top_entropies[i]),
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=7, alpha=0.6)

plt.tight_layout()
plt.show()

# Print some statistics
print(f"Features plotted: {len(top_entropies)}")
print(f"Entropy range: {np.min(top_entropies):.3f} to {np.max(top_entropies):.3f} bits")
print(f"Influence range: {np.min(top_influences):.3f} to {np.max(top_influences):.3f}")